In [1]:
# =========================
# Finance-only baseline model
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# =========================
# 1. Load data
# =========================
# Update the paths if the files are not in your current working directory
X_FIN_PATH = "X_fin_pca.csv"
Y_PATH = "deepseek_Y_quantile_final.csv"

x_fin = pd.read_csv(X_FIN_PATH)
y = pd.read_csv(Y_PATH)

print("x_fin shape:", x_fin.shape)
print("y shape:", y.shape)

print("\nx_fin columns:")
print(x_fin.columns.tolist())

print("\ny columns:")
print(y.columns.tolist())


# =========================
# 2. Keep only the necessary Y columns
# =========================
# Important:
# Do NOT use CAR / MaxDrawdown / Volatility / returns as X features,
# because they were used to construct the label and would cause data leakage.

y_keep = y[["ticker", "label"]].drop_duplicates().copy()

# Convert labels to binary values
# Vulnerable = 1
# Resilient = 0
label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep["label"].map(label_map)

print("\nY label counts:")
print(y_keep["label"].value_counts(dropna=False))
print(y_keep["label_binary"].value_counts(dropna=False))


# =========================
# 3. Merge finance features with Y
# =========================
# x_fin uses 'tic'
# y_keep uses 'ticker'

model_fin = x_fin.merge(
    y_keep,
    left_on="tic",
    right_on="ticker",
    how="inner"
)

print("\nMerged finance-only dataset shape:", model_fin.shape)
print("\nMerged label counts:")
print(model_fin["label"].value_counts(dropna=False))

print("\nDuplicate tickers in merged data:", model_fin["ticker"].duplicated().sum())
print("\nMissing values (top 20):")
print(model_fin.isna().sum().sort_values(ascending=False).head(20))


# =========================
# 4. Define X and y
# =========================
# Use only PCA features: PC1 ~ PC8
finance_feature_cols = [c for c in model_fin.columns if c.startswith("PC")]

print("\nFinance feature columns:")
print(finance_feature_cols)

X = model_fin[finance_feature_cols].copy()
y_binary = model_fin["label_binary"].copy()

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 5. Define evaluation function
# =========================
def evaluate_model(X, y, model_type="logit"):
    """
    Evaluate a model using 5-fold stratified cross-validation.
    Returns accuracy, F1, and ROC-AUC.
    """

    if model_type == "logit":
        # Logistic Regression: scaling is needed
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])
    elif model_type == "rf":
        # Random Forest: scaling is not required
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])
    else:
        raise ValueError("model_type must be either 'logit' or 'rf'")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    result = {
        "model_type": model_type,
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }
    return result


# =========================
# 6. Run two baseline models
# =========================
results = []

results.append(evaluate_model(X, y_binary, model_type="logit"))
results.append(evaluate_model(X, y_binary, model_type="rf"))

results_df = pd.DataFrame(results)

print("\n=== Finance-only baseline results ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 7. Save outputs
# =========================
model_fin.to_csv("model_finance_only.csv", index=False)
results_df.to_csv("baseline_results_finance_only.csv", index=False)

print("\nSaved files:")
print("  model_finance_only.csv")
print("  baseline_results_finance_only.csv")

x_fin shape: (403, 11)
y shape: (158, 17)

x_fin columns:
['tic', 'gvkey', 'datadate', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8']

y columns:
['company_id', 'cik', 'ticker', 'company_name', 'sic', 'sic_description', 'start_date_actual', 'start_price', 'end_date_actual', 'end_price', 'raw_return', 'normal_return_spy', 'excess_return', 'CAR', 'MaxDrawdown', 'Volatility', 'label']

Y label counts:
label
Vulnerable    79
Resilient     79
Name: count, dtype: int64
label_binary
1    79
0    79
Name: count, dtype: int64

Merged finance-only dataset shape: (158, 14)

Merged label counts:
label
Vulnerable    79
Resilient     79
Name: count, dtype: int64

Duplicate tickers in merged data: 0

Missing values (top 20):
tic             0
gvkey           0
datadate        0
PC1             0
PC2             0
PC3             0
PC4             0
PC5             0
PC6             0
PC7             0
PC8             0
ticker          0
label           0
label_binary    0
dtype: int64

Finan